In [1]:
import pandas as pd
import json
import os
import glob

In [2]:
path_to_json = '/Users/srujan/Workspace/Projects/Movie-Recommendation-System/movie_pool/'

def get_files(path):
    json_pattern = os.path.join(path_to_json, f'{path}/*.json')
    file_list = glob.glob(json_pattern)
    return file_list

def get_df(file_list):
    dfs = []

    for file in file_list:
        data = pd.read_json(file)
        dfs.append(data)
    
    return pd.concat(dfs, ignore_index=True)


def remove_duplicates(df):
    return df.drop_duplicates(subset=['id'])


### Genres and Language

In [3]:
genre_file_path = 'genre_master'
language_file_path = 'language_master'




genre = get_df(get_files(path=genre_file_path))
genre_df = pd.json_normalize(genre['genres'])
genre_map = dict(zip(genre_df['id'], genre_df['name']))



language_df = get_df(get_files(path=language_file_path))
language_df = language_df.rename(columns={'iso_639_1': 'iso_code'})
language_map = dict(zip(language_df['iso_code'], language_df['english_name']))



### Popular Movies

In [4]:

path = 'popular_movies'

# flatten the json files
popular_movies = get_df(get_files(path=path))
popular_movies_df = pd.json_normalize(popular_movies['results'].explode())

# remove duplicates and null
popular_movies_df = remove_duplicates(popular_movies_df)
popular_movies_df.dropna(subset=['id'], inplace=True)

# Mapping of genre and language
popular_movies_df['genre_name'] = popular_movies_df['genre_ids'].apply(
    lambda ids: [genre_map[id] for id in ids]
)

popular_movies_df['language'] = popular_movies_df['original_language'].apply(
    lambda iso_code: language_map[iso_code]
)

# select relevant cols
popular_movies_df = popular_movies_df[['id','title', 'overview', 'genre_ids','genre_name', 'original_language','language',  'release_date']]


popular_movies_df



,id,title,overview,genre_ids,genre_name,original_language,language,release_date
0,57737,Barbie: A Fairy Secret,Barbie realises that fairies secretly surround...,"[16, 10751, 14]","[Animation, Family, Fantasy]",en,English,2011-03-01
0,241863,As the Gods Will,High school student Shun Takahata is bored. Bo...,"[53, 27, 35]","[Thriller, Horror, Comedy]",ja,Japanese,2014-11-15
0,513692,Raging Fire,Cheung Sung-bong is a highly respected cop wit...,"[28, 80, 53, 12, 9648]","[Action, Crime, Thriller, Adventure, Mystery]",cn,Cantonese,2021-07-28
0,37347,Fort Apache,Owen Thursday sees his new posting to the deso...,[37],[Western],en,English,1948-05-21
0,901563,Close,Two 13-year-old boys spend an idyllic summer t...,[18],[Drama],fr,French,2022-11-01
...,...,...,...,...,...,...,...,...
499,788929,Lamb,An Icelandic couple live with their herd of sh...,"[18, 14, 27]","[Drama, Fantasy, Horror]",is,Icelandic,2021-08-12
499,939335,Muzzle,LAPD K-9 officer Jake Rosser has just witnesse...,"[28, 53, 10770]","[Action, Thriller, TV Movie]",en,English,2023-09-29
499,183011,Justice League: The Flashpoint Paradox,The Flash finds himself in a war-torn alternat...,"[16, 28, 878]","[Animation, Action, Science Fiction]",en,English,2013-07-30
499,460648,Bleeding Steel,A hardened special forces agent fights to prot...,"[878, 28, 12, 53]","[Science Fiction, Action, Adventure, Thriller]",zh,Mandarin,2017-12-18


### Top Rated Movies

In [5]:
path = 'top_rated_movies'

# flatten the json files
top_rated_movies = get_df(get_files(path=path))
top_rated_movies_df = pd.json_normalize(top_rated_movies['results'].explode())

# remove duplicates and nulls
top_rated_movies_df = remove_duplicates(top_rated_movies_df)
top_rated_movies_df.dropna(subset=['id'], inplace=True)

# Mapping of genre and language
top_rated_movies_df['genre_name'] = top_rated_movies_df['genre_ids'].apply(
    lambda ids: [genre_map[id] for id in ids]
)

top_rated_movies_df['language'] = top_rated_movies_df['original_language'].apply(
    lambda iso_code: language_map[iso_code]
)

# select relevant cols
top_rated_movies_df = top_rated_movies_df[['id','title', 'overview', 'genre_ids','genre_name', 'original_language','language',  'release_date']]

top_rated_movies_df

,id,title,overview,genre_ids,genre_name,original_language,language,release_date
0,589524,The Champion,Christian is an extremely talented as well as ...,"[35, 18]","[Comedy, Drama]",it,Italian,2019-04-18
0,522931,Hitman's Wife's Bodyguard,The world’s most lethal odd couple – bodyguard...,"[28, 35, 80, 53]","[Action, Comedy, Crime, Thriller]",en,English,2021-06-14
0,280180,Beverly Hills Cop: Axel F,Forty years after his unforgettable first case...,"[28, 35, 80]","[Action, Comedy, Crime]",en,English,2024-06-20
0,62835,Colombiana,After witnessing her parents’ murder as a chil...,"[28, 53, 80, 18]","[Action, Thriller, Crime, Drama]",en,English,2011-07-27
0,12155,Alice in Wonderland,"Alice, now 19 years old, returns to the whimsi...","[10751, 14, 12]","[Family, Fantasy, Adventure]",en,English,2010-03-03
...,...,...,...,...,...,...,...,...
499,9093,The Four Feathers,A young British officer resigns his post when ...,"[28, 12, 18, 10749, 10752]","[Action, Adventure, Drama, Romance, War]",en,English,2002-09-08
499,484247,A Simple Favor,"Stephanie, a dedicated mother and popular vlog...","[53, 35, 9648]","[Thriller, Comedy, Mystery]",en,English,2018-09-13
499,50544,Friends with Benefits,Dylan is done with relationships. Jamie decide...,"[10749, 35]","[Romance, Comedy]",en,English,2011-07-21
499,1266,Street Kings,Tom Ludlow is a disillusioned L.A. Police Offi...,"[28, 80, 18, 53]","[Action, Crime, Drama, Thriller]",en,English,2008-04-10


### Now Playing movies

In [6]:
path = 'now_playing_movies'

# flatten the json files
now_playing_movies = get_df(get_files(path=path))

# make data ready before normalizing 
results_series = now_playing_movies['results'].apply(lambda x: x if len(x) > 0 else None).dropna()
now_playing_movies_df = pd.json_normalize(results_series.explode())

# remove duplicates and null
now_playing_movies_df = remove_duplicates(now_playing_movies_df)
now_playing_movies_df.dropna(subset=['id'], inplace=True)


# Mapping of genre and language
now_playing_movies_df['genre_name'] = now_playing_movies_df['genre_ids'].apply(
    lambda ids: [genre_map.get(id) for id in ids]
)

now_playing_movies_df['language'] = now_playing_movies_df['original_language'].apply(
    lambda iso_code: language_map.get(iso_code, 'Unknown')
)

# # select relevant cols
now_playing_movies_df = now_playing_movies_df[['id','title', 'overview', 'genre_ids','genre_name', 'original_language','language',  'release_date']]

# now_playing_movies_df[now_playing_movies_df['original_language'].isna()]
# type(now_playing_movies_df['genre_ids'])

now_playing_movies_df
# now_playing_movies_df[now_playing_movies_df['language']=='Unknown']

,id,title,overview,genre_ids,genre_name,original_language,language,release_date
0,1700728,Virescent Vengeance,2 partners invite a target to a fake meeting t...,"[28, 53, 80]","[Action, Thriller, Crime]",en,English,2026-05-15
0,1700406,I Listen for the Sound of Ashes,I Listen for the Sound of Ashes unfolds throug...,"[18, 10751]","[Drama, Family]",ne,Nepali,2026-05-30
0,1700371,Girl From Nepal,"A girl from Nepal, Hema, comes to Kyrgyzstan a...",[18],[Drama],en,English,2026-05-30
0,1700358,Unreachables,"Bishal decides to meet his friend, Mausam, one...",[18],[Drama],ne,Nepali,2026-05-30
0,1700234,I Have Become a Rhododendron,"Maiya Devi Pandey, a 70-year-old Nepali woman ...",[99],[Documentary],ne,Nepali,2026-05-28
...,...,...,...,...,...,...,...,...
499,1701238,SLO Heist,Special agent Vincent Kane is pulled back into...,"[28, 53, 80]","[Action, Thriller, Crime]",en,English,2026-05-29
499,1701197,Serendipity,Animated short film,[16],[Animation],ja,Japanese,2026-06-05
499,1701124,Sound Of Symbols,Using a 16mm optical audio reader and an oscil...,[],[],en,English,2026-05-29
499,1701049,Best Cinematography,A satire film from Lawson Saby,[],[],en,English,2026-05-14


### Upcoming Movies

In [7]:
path = 'upcoming_movies'

# flatten the json files
upcoming_movies = get_df(get_files(path=path))
upcoming_movies_df = pd.json_normalize(upcoming_movies['results'].explode())


# remove duplicates and null
upcoming_movies_df = remove_duplicates(upcoming_movies_df)
upcoming_movies_df.dropna(subset=['id'], inplace=True)


# # Mapping of genre and language
upcoming_movies_df['genre_name'] = upcoming_movies_df['genre_ids'].apply(
    lambda ids: [genre_map[id] for id in ids]
)

upcoming_movies_df['language'] = upcoming_movies_df['original_language'].apply(
    lambda iso_code: language_map[iso_code]
)

# # select relevant cols
upcoming_movies_df = upcoming_movies_df[['id','title', 'overview', 'genre_ids','genre_name', 'original_language','language',  'release_date']]

upcoming_movies_df

,id,title,overview,genre_ids,genre_name,original_language,language,release_date
25,1704986.0,Where All Windows Meet,,[99],[Documentary],zh,Mandarin,2026-06-13
25,1699981.0,Mnema,"In a world of obstacles and labyrinths, Mnema ...",[99],[Documentary],en,English,2026-06-12
25,1677731.0,We Get Better When We Tan Moose Hides,The tradition of tanning hides with brains was...,[],[],en,English,2026-06-13
25,1639299.0,You're Just Tired,"Feeling isolated, lonely, and bored, new mothe...","[27, 18]","[Horror, Drama]",en,English,2026-06-28
25,1524711.0,The Prayer of The Rose,An emotional journey of a teenage boy asking G...,[10770],[TV Movie],ur,Urdu,2026-06-26
...,...,...,...,...,...,...,...,...
474,1596393.0,Chhaki Suna,"Each victim holds a piece of the puzzle, but s...",[],[],or,Oriya,2026-06-12
474,1409734.0,A Fog of the Fate,"Father Yang unexpectedly passed away, and duri...","[18, 9648, 80]","[Drama, Mystery, Crime]",zh,Mandarin,2026-06-14
474,1689991.0,Alejandra P.,,[],[],es,Spanish,2026-06-20
474,1263313.0,The Son of a Bitch,The quiet city of Veredas is home to Casa Rosa...,[16],[Animation],pt,Portuguese,2026-06-22


### DFS Info

In [8]:
dfs_name = ["popular_movies_df", "top_rated_movies_df", "now_playing_movies_df", "upcoming_movies_df"]
dfs = [popular_movies_df, top_rated_movies_df, now_playing_movies_df, upcoming_movies_df]

df_name_mapping = dict(zip(dfs_name, dfs))

for df_name, df in df_name_mapping.items():
    print(f"df: {df_name}")
    print(f"Total Rows: {len(df)}")
    print("columns info:")
    print(df.columns)
    print("=================")

df: popular_movies_df
Total Rows: 9792
columns info:
Index(['id', 'title', 'overview', 'genre_ids', 'genre_name',
       'original_language', 'language', 'release_date'],
      dtype='str')
df: top_rated_movies_df
Total Rows: 9710
columns info:
Index(['id', 'title', 'overview', 'genre_ids', 'genre_name',
       'original_language', 'language', 'release_date'],
      dtype='str')
df: now_playing_movies_df
Total Rows: 6056
columns info:
Index(['id', 'title', 'overview', 'genre_ids', 'genre_name',
       'original_language', 'language', 'release_date'],
      dtype='str')
df: upcoming_movies_df
Total Rows: 1148
columns info:
Index(['id', 'title', 'overview', 'genre_ids', 'genre_name',
       'original_language', 'language', 'release_date'],
      dtype='str')


### Combined movie pool

In [9]:
movies_df = pd.concat([popular_movies_df, top_rated_movies_df, now_playing_movies_df, upcoming_movies_df])

In [10]:
len(movies_df)

26706

In [11]:
movies_df.head(5)

,id,title,overview,genre_ids,genre_name,original_language,language,release_date
0,57737.0,Barbie: A Fairy Secret,Barbie realises that fairies secretly surround...,"[16, 10751, 14]","[Animation, Family, Fantasy]",en,English,2011-03-01
0,241863.0,As the Gods Will,High school student Shun Takahata is bored. Bo...,"[53, 27, 35]","[Thriller, Horror, Comedy]",ja,Japanese,2014-11-15
0,513692.0,Raging Fire,Cheung Sung-bong is a highly respected cop wit...,"[28, 80, 53, 12, 9648]","[Action, Crime, Thriller, Adventure, Mystery]",cn,Cantonese,2021-07-28
0,37347.0,Fort Apache,Owen Thursday sees his new posting to the deso...,[37],[Western],en,English,1948-05-21
0,901563.0,Close,Two 13-year-old boys spend an idyllic summer t...,[18],[Drama],fr,French,2022-11-01


In [12]:
temp_df = movies_df.drop_duplicates(subset=['id'])

len(temp_df)

20628